In [ ]:
from common import *

In [2]:
def optimize(params, x, y):
    model = ensemble.RandomForestClassifier(**params)
    kf = model_selection.StratifiedKFold(n_splits=5)
    
    accuracies = []
    for idx in kf.split(x, y):
        train_idx,  val_idx         = idx[0],       idx[1]
        train_x,    train_y         = x[train_idx], y[train_idx]
        val_x,      val_y           = x[val_idx],   y[val_idx]
        
        model.fit(train_x, train_y)
        val_preds = model.predict(val_x)
        fold_acc = metrics.accuracy_score(val_y, val_preds)
        accuracies.append(fold_acc)
    
    # -1 because we minimize this
    return -1.0 * np.mean(accuracies)

In [3]:
param_space = {
                        # "quantized" uniform distribution
           "max_depth": scope.int(hp.quniform ("max_depth",       3,  15, 1)),
        "n_estimators": scope.int(hp.quniform ("n_estimators",  100, 600, 1)),
        "max_features": hp.uniform  ("max_features", 0.01,   1),
           "criterion": hp.choice   ("criterion", ["gini", "entropy"]),
}

# fix all arguments except params
optimization_func = partial(
    optimize,
    x=X,
    y=y
)

In [4]:
trials = Trials()

result = fmin(
    optimization_func,
    algo=tpe.suggest,
    space=param_space,
    max_evals=15,
    trials=trials,
    verbose=10,
)

100%|██████████| 15/15 [01:26<00:00,  5.79s/trial, best loss: -0.9095000000000001]


In [6]:
result

{'criterion': np.int64(1),
 'max_depth': np.float64(13.0),
 'max_features': np.float64(0.9768149352378446),
 'n_estimators': np.float64(292.0)}